# Exact realizability checker

Checks whether a set of **safety** goals, given as automata in HOA format (as exported by Spot),
is standardly or exactly realizable, within a domain theory DT. It also finds all minimal conflicts: sets of goals that are not realizable together, while every strict subset is.

- **Standard:** some controller satisfies the goals (it may be more restrictive than they are, e.g. avoid situations).
- **Exact:** some input-enabled controller has exactly the traces of the goals; it cannot avoid anything they allow.

Steps:

- 1. Each LTL formula of a spec is translated by Spot into a deterministic automaton with only accepting states (HOA).

- 2. Check realizability

  - 2.1 **exact** = every reachable state is input-enabled, computed on the
  determinized automaton;

  - 2.2 **standard** = a safety game (greatest fixpoint of winning states).

**Timing semantics** (a parameter, and it matters):

- `moore`: at each step the actor commits its outputs before seeing that step's inputs
  (`ap_state` is the mode the system is *in* when the inputs arrive);
- `mealy`: the actor sees the inputs first, then picks the outputs (GR(1)/Spectra style).

The specs are in `specs/*.toml`. If Spot is not installed, the notebook uses the reference automata in `examples/hoa/`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))      # not needed after `pip install -e .`

from exactreal import *
print('Spot:', spot_backend() or 'not installed (using examples/hoa)')

def load_example(name, with_assumptions=False):
    spec = load_spec(ROOT / 'specs' / f'{name}.toml')
    automata, where = build_automata(spec, ROOT / 'examples' / 'hoa' / name)
    goals, dom = goals_and_domain(spec, automata, with_assumptions)
    print(f'{spec.name}: automata from', where.replace(str(ROOT) + '/', ''))
    return spec, goals, dom

def show_conflicts(goals, dom):
    for notion in ['standard', 'exact']:
        for sem in ['mealy', 'moore']:
            cs = minimal_conflicts(goals, dom, notion, sem)
            print(f'{notion:8} {sem:5}: {len(cs):2} conflicts, sizes {sorted({len(c) for c in cs})}')
            if len(cs) <= 8:
                for c in cs:
                    print('            ', ', '.join(c))

Spot: not installed (using examples/hoa)


## Mine pump

In [2]:
spec, MP, MP_DT = load_example('mine_pump')
R1, R2, R3 = MP
print(check([R1, R2], MP_DT))

Mine pump: automata from files in examples/hoa/mine_pump
Goals R1, R2  |  semantics: moore
  inputs  (env)  : ['highWater', 'lowWater', 'methane']
  outputs (actor): ['PUMP']
  standard realizable: False
  exactly realizable : False
  why not exact:
    at the initial step
    then the actor may output {!PUMP}, but env {highWater, !lowWater, methane} would violate the goals


In [3]:
show_conflicts(MP, MP_DT)
_, MP_A, MP_A_DT = load_example('mine_pump', with_assumptions=True)
print('with the assumption !methane in the domain (diagnostic):', minimal_conflicts(MP_A, MP_A_DT))

standard mealy:  1 conflicts, sizes [2]
             R1, R2
standard moore:  1 conflicts, sizes [2]
             R1, R2
exact    mealy:  1 conflicts, sizes [2]
             R1, R2
exact    moore:  1 conflicts, sizes [2]
             R1, R2
Mine pump: automata from files in examples/hoa/mine_pump
with the assumption !methane in the domain (diagnostic): []


## Autopilot (FRET)

In [4]:
spec, AUTO, AUTO_DT = load_example('autopilot')
for sem in ['moore', 'mealy']:
    print(check(AUTO[:2], AUTO_DT, sem), '\n')
show_conflicts(AUTO, AUTO_DT)

Autopilot (FRET): automata from files in examples/hoa/autopilot
Goals R02, R03  |  semantics: moore
  inputs  (env)  : ['good', 'standby', 'supported']
  outputs (actor): ['AP_MANEUVER', 'AP_NOMINAL', 'AP_STANDBY', 'AP_TRANSITION']
  standard realizable: True
  exactly realizable : False
  why not exact:
    at the initial step
    then the actor may output {!AP_MANEUVER, !AP_NOMINAL, !AP_STANDBY, AP_TRANSITION}, but env {good, standby, supported} would violate the goals 

Goals R02, R03  |  semantics: mealy
  inputs  (env)  : ['good', 'standby', 'supported']
  outputs (actor): ['AP_MANEUVER', 'AP_NOMINAL', 'AP_STANDBY', 'AP_TRANSITION']
  standard realizable: True
  exactly realizable : True 



standard mealy:  8 conflicts, sizes [5]
             R02, R03, R05, R07, R09
             R02, R05, R06, R07, R09
             R02, R05, R06, R08, R09
             R02, R05, R07, R08, R09
             R03, R04, R05, R07, R09
             R03, R05, R06, R07, R09
             R03, R05, R06, R08, R09
             R03, R05, R07, R08, R09


standard moore: 21 conflicts, sizes [5]


exact    mealy:  8 conflicts, sizes [3]
             R02, R03, R07
             R02, R03, R08
             R02, R08, R09
             R03, R04, R05
             R04, R06, R07
             R05, R08, R09
             R06, R07, R09
             R06, R08, R09


exact    moore:  4 conflicts, sizes [2]
             R02, R03
             R04, R05
             R06, R07
             R08, R09


- **Standard + Mealy reproduces Spectra**: 8 unrealizable cores of size 5, as Alex found.
- **Exact + Moore gives the 4 expected binary conflicts**: R02/R03, R04/R05, R06/R07, R08/R09.
- **Exact + Mealy gives triples instead.**

## Liquid mixer

In [5]:
spec, LM, LM_DT = load_example('liquid_mixer')
for e in spec.entries:
    print(f'{e.id:4} {e.kind:11} {e.ltl}')

Liquid mixer: automata from files in examples/hoa/liquid_mixer
R01  requirement G(start -> X(VALVE_0 & !VALVE_1 & !VALVE_2 & !MIXER))
R02  requirement G((level_1_reached & !stop) -> X(!VALVE_0 & VALVE_1 & !VALVE_2 & !MIXER))
R03  requirement G((level_2_reached & !stop) -> X(!VALVE_0 & !VALVE_1 & !VALVE_2 & TIMER_60))
R04  requirement G((level_2_reached & !stop) -> X(MIXER W (timer60_ends | stop)))
R05  requirement G((timer60_ends & !stop) -> X(!VALVE_0 & !VALVE_1 & !MIXER & TIMER_120))
R06  requirement G((timer60_ends & !stop) -> X(VALVE_2 W (timer120_ends | stop)))
R07  requirement G(timer120_ends -> X(!VALVE_0 & !VALVE_1 & !VALVE_2 & !MIXER))
R08  requirement G(stop -> X(!VALVE_0 & !VALVE_1 & !VALVE_2 & !MIXER))
D01  domain      G !(level_1_reached & level_2_reached)
A01  assumption  G(!VALVE_0 -> X !level_1_reached)
A02  assumption  G(!VALVE_1 -> X !level_2_reached)
A03  assumption  (!timer60_ends W TIMER_60) & G(timer60_ends -> X(!timer60_ends W TIMER_60))
A04  assumption  (!timer1

In [6]:
show_conflicts(LM, LM_DT)

standard mealy: 15 conflicts, sizes [2]


standard moore: 15 conflicts, sizes [2]


exact    mealy: 15 conflicts, sizes [2]


exact    moore:  9 conflicts, sizes [1, 2]


In [7]:
pick = lambda goals, *names: [g for g in goals if g.name in names]
print(check(pick(LM, 'R01', 'R08'), LM_DT, 'mealy'))

Goals R01, R08  |  semantics: mealy
  inputs  (env)  : ['level_1_reached', 'level_2_reached', 'start', 'stop']
  outputs (actor): ['MIXER', 'VALVE_0', 'VALVE_1', 'VALVE_2']
  standard realizable: False
  exactly realizable : False
  why not exact:
    at the initial step
    then env {!level_1_reached, !level_2_reached, start, stop} leaves the actor no valid output


Which conflicts would the assumptions hide? (Diagnostic only: assumptions are not part of the domain theory.)

In [8]:
_, LM_A, LM_A_DT = load_example('liquid_mixer', with_assumptions=True)
print('exact mealy, with A01-A05 in the domain:', minimal_conflicts(LM_A, LM_A_DT, 'exact', 'mealy'))

Liquid mixer: automata from files in examples/hoa/liquid_mixer


exact mealy, with A01-A05 in the domain: [['R01', 'R02'], ['R01', 'R03'], ['R01', 'R04'], ['R01', 'R05'], ['R01', 'R06'], ['R01', 'R07'], ['R02', 'R04'], ['R02', 'R05'], ['R02', 'R06'], ['R02', 'R07'], ['R03', 'R06'], ['R04', 'R05'], ['R04', 'R07'], ['R06', 'R07']]


## Patient monitoring

In [9]:
spec, PM, PM_DT = load_example('patient_monitoring')
show_conflicts(PM, PM_DT)

Patient monitoring: automata from files in examples/hoa/patient_monitoring
standard mealy:  0 conflicts, sizes []
standard moore:  0 conflicts, sizes []
exact    mealy:  0 conflicts, sizes []
exact    moore:  0 conflicts, sizes []
